<a href="https://colab.research.google.com/github/Arsh-Vora/Arxiv-PageRank-Analysis/blob/dev/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("Cornell-University/arxiv")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'arxiv' dataset.
Path to dataset files: /kaggle/input/arxiv


In [6]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("arXiv Metadata").getOrCreate()

In [7]:
schema = StructType([
    StructField("id", StringType(), True),
    StructField("authors_parsed", ArrayType(ArrayType(StringType())), True)
])

raw_df = spark.read.json(path + "/arxiv-metadata-oai-snapshot.json", schema=schema)

df_processed = raw_df.withColumn(
    "author_list",
    F.expr("""
        transform(authors_parsed, x ->
            regexp_replace(trim(concat(x[1], ' ', x[0])), "\\\\\\\\[\'`^\\\"~]", "")
        )
    """)

)

authors_exploded = df_processed.select(
    "id",
    F.explode("author_list").alias("author")
)

edges = authors_exploded.alias("a").join(
    authors_exploded.alias("b"), "id"
).filter("a.author != b.author") \
 .select(F.col("a.author").alias("source"), F.col("b.author").alias("target")) \
 .distinct()

edges.cache()
print(f"Graph initialized with {edges.count()}unique edges")

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 